# The electric field in a capacitor

The electrostatic potential $\phi$ satisfies

$$
-\operatorname{div} (\varepsilon \nabla \phi) = 0 \quad \text{in } \Omega, \qquad \phi = \pm 1 \text{V} \quad \text{on the plates}.
$$

Multiply by a test function $\psi$ vanishing on the plates, integrate over $\Omega$, and integrate by parts:

$$
\int_\Omega \varepsilon \nabla \phi \cdot \nabla \psi \, dx = 0 \qquad \forall \, \psi.
$$

Only first derivatives remain, so continuous piecewise polynomials are good enough: taking $\phi$ and $\psi$ from a finite element space $V_h \subset H^1(\Omega)$ gives a sparse linear system.
Equivalently, $\phi$ minimizes the field energy $\tfrac{1}{2} \int \varepsilon |\nabla \phi|^2 dx$, a view which carries over to nonlinear problems.
In NGSolve, the problem is written as it is on paper: `Solve(eps*grad(u)*grad(v)*dx == 0, u[BND('el1')]==1, ...)`.

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

## Sharp plates: hp-refinement

Plates are holes in the air box, their boundaries are named for the boundary conditions.
At the sharp corners the electric field is singular. The mesh is refined geometrically towards the corners (hp-FEM), which restores exponential convergence:

In [ ]:
def MakeMesh():
    shape = MoveTo(0,0).RectangleC(20,20) \
        .MoveTo(0,1).RectangleC(5,0.5, "el1").Reverse() \
        .MoveTo(0,-1).RectangleC(5,0.5, "el2").Reverse() \
        .Face()
    shape.edges["el.*"].vertices.hpref=1
    mesh = shape.GenerateMesh(dim=2, maxh=2)
    return shape,mesh

In [ ]:
geo, mesh = MakeMesh()
mesh.RefineHP(3, 0.25)
Draw (geo)
Draw (mesh);

Solve the variational problem. The plate potentials +1 Volt and -1 Volt enter as inhomogeneous Dirichlet conditions:

In [ ]:
eps0 = 8.854e-12

def SolveProblem(mesh):
    u,v = H1(mesh, order=5).TnT()
    return Solve(eps0*grad(u)*grad(v)*dx == 0, u[BND('el1')]==1, u[BND('el2')]==-1)

gfu = SolveProblem(mesh)

ea = { "euler_angles" : [-70,0,-40]} 
Draw (gfu, deformation=True, scale=5, **ea);

The electric field $E = -\nabla \phi$, resolved up to the corners:

In [ ]:
Draw (Norm(grad(gfu)), mesh, order=3, deformation=True, min=0, max=3, **ea);

## Rounded plates: curved elements

Rounded plates avoid the singularity. Elements are curved to follow the exact CAD geometry (here polynomials of order 5); straight-sided elements would create artificial field peaks at the vertices of the polygonal boundary:

In [ ]:
def MakeMeshRounded():
    square = MoveTo(0,0).RectangleC(20,20).Face()
    el1 = MoveTo(0,1).RectangleC(5,0.5).Face()
    el1 += MoveTo(2.5,1).Circle(0.25).Face()
    el1 += MoveTo(-2.5,1).Circle(0.25).Face()
    el1.edges.name="el1"
    el2 = MoveTo(0,-1).RectangleC(5,0.5).Face()
    el2 += MoveTo(2.5,-1).Circle(0.25).Face()
    el2 += MoveTo(-2.5,-1).Circle(0.25).Face()
    el2.edges.name="el2"
    geo = square - el1 - el2
    mesh = geo.GenerateMesh(dim=2, maxh=2)
    return geo,mesh

In [ ]:
geo, mesh = MakeMeshRounded()
mesh.Curve(5)
Draw (geo)
Draw (mesh);

In [ ]:
gfu = SolveProblem(mesh)
Draw (gfu, deformation=True, scale=5, **ea)
Draw (Norm(grad(gfu)), mesh, order=3, deformation=True, min=0, max=3, **ea);